## A/B Testing Analysis - E-commerce Dataset

This notebook analyzes an A/B experiment to evaluate whether the treatment group performs better than the control group in terms of conversion and revenue.


### Data Loading

The dataset consists of multiple tables including customer information, user events, transactions, products, and marketing campaigns.

These tables are loaded to build a unified analytical dataset for A/B testing.


In [2]:
import pandas as pd

customers = pd.read_csv('data/customers.csv')
events = pd.read_csv('data/events.csv')
transactions = pd.read_csv('data/transactions.csv')
products = pd.read_csv('data/products.csv')
campaigns = pd.read_csv('data/campaigns.csv')

### Data Exploration

Initial inspection of datasets to understand structure, columns, and data quality before transformation.


In [3]:
customers.head()
events.head()
transactions.head()

,transaction_id,timestamp,customer_id,product_id,quantity,discount_applied,gross_revenue,campaign_id,refund_flag
0,1,2021-12-27 08:25:15,59540,1630.0,3,0.00,43.74,0,0
1,2,2023-06-06 21:14:26,54871,1901.0,3,0.00,174.78,21,0
2,3,2023-08-31 05:29:54,51818,1884.0,1,0.00,40.61,37,0
3,4,2022-06-26 20:33:46,18164,1114.0,2,0.15,68.76,13,0
4,5,2023-07-26 18:12:35,86915,408.0,1,0.00,14.64,4,0


### Conversion Flag Creation

A conversion variable is created to indicate whether a user has completed a purchase.

* 1 → User made a purchase
* 0 → No purchase

This is the core metric for A/B testing.


In [4]:
transactions['converted'] = 1

### Data Integration

User behavior data (events) is merged with transaction data to identify which users converted.

This step creates a unified dataset containing:

* Experiment group (A/B)
* User behavior
* Conversion outcome


In [5]:
df = events.merge(
    transactions[['customer_id', 'converted']],
    on='customer_id',
    how='left'
)

### Handling Missing Values

Users without transactions are labeled as non-converted (0).

This ensures a complete dataset for accurate conversion rate calculation.


In [6]:
df['converted'] = df['converted'].fillna(0)

### Data Validation

Final dataset is validated to ensure:

* Each user has an experiment group
* Conversion flag is correctly assigned

This dataset will be used for A/B testing analysis.


In [7]:
df[['customer_id','experiment_group','converted']].head()

,customer_id,experiment_group,converted
0,43812,Control,1.0
1,43812,Control,1.0
2,43812,Control,1.0
3,43812,Control,1.0
4,71340,Variant_A,1.0


### User-Level Data Transformation

The dataset currently contains multiple rows per user due to multiple events.

For A/B testing, analysis must be performed at the **user level**, where each user is represented only once.

This step aggregates the data to create a single record per user.


In [11]:
df_user = df.groupby('customer_id').agg({
    'experiment_group': 'first',
    'converted': 'max'
}).reset_index()

### Aggregation Logic

* **experiment_group (first)** → Each user belongs to a single experiment group
* **converted (max)** → If a user made at least one purchase, they are marked as converted (1)

This ensures that:

* Each user appears only once
* Conversion reflects user-level behavior



### Data Validation

Previewing the transformed dataset to confirm that each user has:

* A single experiment group
* A correct conversion flag


In [13]:
df_user.head()

,customer_id,experiment_group,converted
0,1,Control,0.0
1,2,Control,1.0
2,3,Control,0.0
3,4,Control,0.0
4,5,Control,0.0


### Conversion Rate by Experiment Group

Calculating conversion rates for Control and Variant groups to evaluate experiment performance.


In [14]:
df_user.groupby('experiment_group')['converted'].mean()

experiment_group
Control      0.639507
Variant_A    0.642398
Variant_B    0.640827
Name: converted, dtype: float64

### Group Distribution

Checking the number of users in each experiment group to ensure balanced groups.


In [15]:
df_user['experiment_group'].value_counts()

Control      60015
Variant_A    20067
Variant_B    19918
Name: experiment_group, dtype: int64

### Experiment Groups Overview

The dataset includes three groups:

* Control (baseline)
* Variant_A
* Variant_B

This represents a multi-variant experiment where multiple versions are tested against the control group.

Initial results show that Variant_A has the highest conversion rate, although differences between groups appear relatively small.


### Lift Calculation

Lift measures the relative improvement of a variant compared to the control group.

It helps quantify how much better (or worse) a variant performs in terms of conversion rate.


In [16]:
cr = df_user.groupby('experiment_group')['converted'].mean()

lift_A = (cr['Variant_A'] - cr['Control']) / cr['Control']
lift_B = (cr['Variant_B'] - cr['Control']) / cr['Control']

print("Lift Variant_A:", lift_A)
print("Lift Variant_B:", lift_B)

Lift Variant_A: 0.004520947841925941
Lift Variant_B: 0.002065032553218048


In [17]:
summary = pd.DataFrame({
    'Conversion Rate': cr,
})

summary.loc['Variant_A', 'Lift vs Control'] = lift_A
summary.loc['Variant_B', 'Lift vs Control'] = lift_B

summary

,Conversion Rate,Lift vs Control
experiment_group,,
Control,0.639507,NaN
Variant_A,0.642398,0.004521
Variant_B,0.640827,0.002065


### Key Findings

* Variant_A achieved the highest conversion rate among all groups.
* Variant_B also outperformed the Control group, but with a smaller uplift.
* However, the observed improvements are relatively small (below 1%).

This indicates that while the variants perform slightly better, the practical impact on business outcomes may be limited.


### Statistical Significance Test (Z-Test)

To determine whether the observed differences between groups are statistically significant, a proportion Z-test is performed.

This test evaluates whether the difference in conversion rates is due to chance or represents a real effect.


In [18]:
from statsmodels.stats.proportion import proportions_ztest

In [19]:
# başarı sayısı (converted = 1 olanlar)
success = [
    df_user[df_user['experiment_group'] == 'Control']['converted'].sum(),
    df_user[df_user['experiment_group'] == 'Variant_A']['converted'].sum()
]

# toplam user sayısı
nobs = [
    df_user[df_user['experiment_group'] == 'Control']['converted'].count(),
    df_user[df_user['experiment_group'] == 'Variant_A']['converted'].count()
]

stat, p_value = proportions_ztest(success, nobs)

print("Z-stat:", stat)
print("P-value:", p_value)

Z-stat: -0.7387511709926174
P-value: 0.4600581070534546


In [20]:
success = [
    df_user[df_user['experiment_group'] == 'Control']['converted'].sum(),
    df_user[df_user['experiment_group'] == 'Variant_B']['converted'].sum()
]

nobs = [
    df_user[df_user['experiment_group'] == 'Control']['converted'].count(),
    df_user[df_user['experiment_group'] == 'Variant_B']['converted'].count()
]

stat, p_value = proportions_ztest(success, nobs)

print("Z-stat:", stat)
print("P-value:", p_value)

Z-stat: -0.3364164722572114
P-value: 0.7365568295704917


### Statistical Test Results

The statistical tests show that:

* The difference between Control and Variant_A is not statistically significant (p-value = 0.46)
* The difference between Control and Variant_B is also not statistically significant (p-value = 0.73)

This indicates that the observed differences in conversion rates may be due to random variation rather than a true effect.


### Final Recommendation

Although Variant_A shows a slightly higher conversion rate, the improvement is not statistically significant.

From a business perspective, this means:

* There is no strong evidence that the new variants outperform the control
* Rolling out the new version may not provide meaningful benefit

It is recommended to:

* Run further experiments
* Test stronger variations
* Or increase sample size for more reliable results
